In [1]:
import pandas as pd

df = pd.read_csv(
    'data/phenotype.hpoa',
    sep='\t',
    comment='#',
    low_memory=False
)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Shape: (282723, 12)

Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']


In [2]:
df.head(3)

,database_id,disease_name,qualifier,hpo_id,reference,evidence,onset,frequency,sex,modifier,aspect,biocuration
0,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0011097,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
1,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0002187,PMID:31675180,PCS,NaN,1/1,NaN,NaN,P,HPO:probinson[2021-06-21]
2,OMIM:619340,Developmental and epileptic encephalopathy 96,NaN,HP:0001518,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]


In [3]:
marfan = df[df['database_id'] == 'OMIM:154700']
print(f"Number of annotations: {len(marfan)}")
marfan[['database_id', 'disease_name', 'hpo_id', 'frequency', 'aspect', 'biocuration']].head(10)

Number of annotations: 71


,database_id,disease_name,hpo_id,frequency,aspect,biocuration
94237,OMIM:154700,Marfan syndrome,HP:0430043,16/21,P,HPO:probinson[2024-08-04];HPO:probinson[2024-0...
94238,OMIM:154700,Marfan syndrome,HP:0000483,3/53,P,HPO:probinson[2021-04-01]
94239,OMIM:154700,Marfan syndrome,HP:0001377,29/199,P,HPO:probinson[2021-05-27];HPO:probinson[2021-0...
94240,OMIM:154700,Marfan syndrome,HP:0000486,110/573,P,HPO:skoehler[2015-07-26];HPO:probinson[2020-08...
94241,OMIM:154700,Marfan syndrome,HP:0005136,NaN,P,HPO:probinson[2012-04-24]
94242,OMIM:154700,Marfan syndrome,HP:0001371,NaN,P,HPO:probinson[2012-04-24]
94243,OMIM:154700,Marfan syndrome,HP:0003199,NaN,P,HPO:probinson[2012-04-24]
94244,OMIM:154700,Marfan syndrome,HP:0025586,8/573,P,HPO:skoehler[2018-10-08];HPO:probinson[2021-05...
94245,OMIM:154700,Marfan syndrome,HP:0000518,118/199,P,HPO:probinson[2012-04-24];HP:probinson[2018-09...
94246,OMIM:154700,Marfan syndrome,HP:0008132,NaN,P,HPO:probinson[2012-04-24]


In [4]:
print("Unique diseases:", df['database_id'].nunique())
print("\nBreakdown by database:")
print(df['database_id'].str.split(':').str[0].value_counts())

Unique diseases: 12996

Breakdown by database:
database_id
OMIM        166574
ORPHA       115853
DECIPHER       296
Name: count, dtype: int64


In [5]:
# See what the biocuration column actually looks like
df['biocuration'].dropna().head(10).tolist()

['HPO:probinson[2021-06-21]',
 'HPO:probinson[2021-06-21]',
 'HPO:probinson[2021-06-21]',
 'HPO:probinson[2021-06-21]',
 'HPO:probinson[2021-06-21]',
 'HPO:probinson[2021-06-21]',
 'HPO:probinson[2021-06-21]',
 'HPO:probinson[2021-06-21]',
 'HPO:probinson[2021-06-21]',
 'HPO:probinson[2021-06-21];HPO:probinson[2021-06-21]']

In [6]:
import re
from datetime import date

def extract_creation_date(biocuration_str):
    """
    Extracts the first (creation) date from a biocuration string.
    Input:  'HPO:probinson[2021-03-14];HPO:lhamilton[2023-09-12]'
    Output: date(2021, 3, 14)
    """
    if pd.isna(biocuration_str):
        return None
    match = re.search(r'\[(\d{4}-\d{2}-\d{2})\]', str(biocuration_str))
    if match:
        return pd.to_datetime(match.group(1)).date()
    return None

# Test it manually before applying to the whole dataframe
test_cases = [
    'HPO:probinson[2021-03-14]',
    'HPO:probinson[2019-06-01];HPO:lhamilton[2023-09-12]',
    None,
    ''
]
for t in test_cases:
    print(f"Input: {t!r:55} → Output: {extract_creation_date(t)}")

Input: 'HPO:probinson[2021-03-14]'                             → Output: 2021-03-14
Input: 'HPO:probinson[2019-06-01];HPO:lhamilton[2023-09-12]'   → Output: 2019-06-01
Input: None                                                    → Output: None
Input: ''                                                      → Output: None


In [7]:
df['creation_date'] = df['biocuration'].apply(extract_creation_date)

# Check how many rows have a parseable date
print("Rows with a date:", df['creation_date'].notna().sum())
print("Rows without:    ", df['creation_date'].isna().sum())

# Preview
df[['biocuration', 'creation_date']].dropna().head(5)

Rows with a date: 282716
Rows without:     7


,biocuration,creation_date
0,HPO:probinson[2021-06-21],2021-06-21
1,HPO:probinson[2021-06-21],2021-06-21
2,HPO:probinson[2021-06-21],2021-06-21
3,HPO:probinson[2021-06-21],2021-06-21
4,HPO:probinson[2021-06-21],2021-06-21


In [8]:
print("Frequency values used in HPOA:")
print(df['frequency'].value_counts().head(20))

Frequency values used in HPOA:
frequency
HP:0040283    45225
HP:0040282    40048
HP:0040281    25797
1/1           18123
2/2            7596
HP:0040284     6942
1/2            6193
3/3            4244
1/3            4142
1/4            3363
4/4            2929
1/5            2342
2/3            1990
5/5            1810
1/6            1777
2/4            1542
1/7            1469
6/6            1306
2/5            1223
3/4            1136
Name: count, dtype: int64


In [9]:
# The main frequency HPO terms you need to know:
freq_map = {
    'HP:0040280': 'Obligate (100%)',
    'HP:0040281': 'Very frequent (80-99%)',
    'HP:0040282': 'Frequent (30-79%)',
    'HP:0040283': 'Occasional (5-29%)',
    'HP:0040284': 'Very rare (<4-5%)',
    'HP:0040285': 'Excluded (0%)',
}

for term, label in freq_map.items():
    count = (df['frequency'] == term).sum()
    print(f"{term}  {label:30}  → {count:,} annotations")

HP:0040280  Obligate (100%)                 → 650 annotations
HP:0040281  Very frequent (80-99%)          → 25,797 annotations
HP:0040282  Frequent (30-79%)               → 40,048 annotations
HP:0040283  Occasional (5-29%)              → 45,225 annotations
HP:0040284  Very rare (<4-5%)               → 6,942 annotations
HP:0040285  Excluded (0%)                   → 0 annotations


In [10]:
# Filter to just obligate + very frequent phenotype annotations across all OMIM diseases
hallmarks_all = df[
    (df['frequency'].isin(['HP:0040280', 'HP:0040281'])) &
    (df['aspect'] == 'P') &
    (df['database_id'].str.startswith('OMIM:'))
]

hallmarks_per_disease = (
    hallmarks_all
    .groupby(['database_id', 'disease_name'])
    .size()
    .reset_index(name='hallmark_count')
    .sort_values('hallmark_count', ascending=False)
)

print("Distribution of hallmark counts per disease:")
print(hallmarks_per_disease['hallmark_count'].describe())
print("\nTop 10 most annotated diseases:")
print(hallmarks_per_disease.head(10).to_string(index=False))

Distribution of hallmark counts per disease:
count    57.000000
mean      2.614035
std       3.080784
min       1.000000
25%       1.000000
50%       1.000000
75%       3.000000
max      17.000000
Name: hallmark_count, dtype: float64

Top 10 most annotated diseases:
database_id                                                                    disease_name  hallmark_count
OMIM:176270                                                           Prader-Willi syndrome              17
OMIM:200500                                                                     Acheiropody              15
OMIM:200990                                                           Acrocallosal syndrome               7
OMIM:276820                        Ulna and fibula, absence of, with severe limb deficiency               7
OMIM:184460                                      Stapes ankylosis with broad thumb and toes               6
OMIM:194190                                                        Wolf-Hirschhorn sy